# 04 — STEM Regional Analysis

Notebook ini menganalisis variasi regional keluaran lulusan STEM dan non-STEM pada data pendidikan tinggi Indonesia tahun 2025.

Tiga konstruk dipisahkan secara tegas:

1. **Jumlah lulusan STEM** — ukuran volume keluaran.
2. **Proporsi STEM dalam seluruh lulusan** — ukuran komposisi.
3. **Graduate-to-entry flow ratio** — jumlah lulusan relatif terhadap mahasiswa baru pada periode yang sama.

> Rasio aliran bukan *completion rate*, *graduation probability*, atau ukuran retensi cohort karena mahasiswa baru dan lulusan tahun 2025 bukan individu/cohort yang sama.

## Model inferensial

### Model komposisi

Pada strata wilayah–jenjang yang memiliki pasangan STEM dan non-STEM:

\[
(Y_{STEM},Y_{NON-STEM}) \sim \operatorname{Binomial}(N,p_{STEM}).
\]

Model ini menjelaskan odds seorang lulusan agregat berada dalam kategori STEM pada strata tersebut. Standard error dikelompokkan menurut provinsi.

### Model aliran jumlah

\[
Y_i \sim \operatorname{NB2}(\mu_i,\alpha),\qquad
\log(\mu_i)=\log(\text{mahasiswa baru}_i)+X_i\beta.
\]

Model ini membandingkan expected graduate output relatif terhadap arus mahasiswa baru, bukan peluang individu menyelesaikan studi.

## Batasan

- Data bersifat cross-sectional dan ekologis.
- Hubungan statistik bukan bukti kausal.
- Definisi STEM mengikuti klasifikasi dalam dataset nasional.
- Analisis regional belum memasukkan ketetanggaan geografis karena shapefile/kode BPS belum tersedia.

In [1]:
from __future__ import annotations

import json
import math
import platform
import random
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import patsy
import scipy
import statsmodels
import statsmodels.api as sm
import statsmodels.formula.api as smf
from IPython.display import display
from scipy import stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_squared_log_error, mean_poisson_deviance
from statsmodels.discrete.discrete_model import NegativeBinomial

warnings.filterwarnings('ignore', category=FutureWarning)
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

pd.set_option('display.max_columns', 120)
pd.set_option('display.max_rows', 120)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

print('Python      :', sys.version.split()[0])
print('Pandas      :', pd.__version__)
print('NumPy       :', np.__version__)
print('SciPy       :', scipy.__version__)
print('Statsmodels :', statsmodels.__version__)
print('Platform    :', platform.platform())

Python      : 3.13.5
Pandas      : 2.2.3
NumPy       : 2.3.5
SciPy       : 1.17.0
Statsmodels : 0.14.6
Platform    : Linux-6.12.13-x86_64-with-glibc2.41


In [2]:
CURRENT_DIR = Path.cwd().resolve()
WORKBOOK_NAME = 'Dataset_Gabungan_Kemdiktisaintek_2025.xlsx'


def candidate_roots(start: Path) -> list[Path]:
    out=[]
    def add(p):
        try: p=p.resolve()
        except OSError: return
        if p not in out: out.append(p)
    add(start)
    for p in start.parents: add(p)
    try:
        for child in start.iterdir():
            if child.is_dir(): add(child)
    except (FileNotFoundError, PermissionError):
        pass
    for base in list(out):
        for name in [
            'pt_education_research_project',
            'pt_education_research_project_00_01_02',
            'pt_education_research_project_00_01_02_04',
        ]:
            add(base/name)
    return out

PROJECT_ROOT=None
COMBINED_FILE=None
for root in candidate_roots(CURRENT_DIR):
    candidate=root/'data'/'processed'/WORKBOOK_NAME
    if candidate.exists():
        PROJECT_ROOT=root; COMBINED_FILE=candidate; break
if COMBINED_FILE is None:
    for root in [CURRENT_DIR, CURRENT_DIR.parent]:
        candidate=root/WORKBOOK_NAME
        if candidate.exists():
            PROJECT_ROOT=root; COMBINED_FILE=candidate; break
if PROJECT_ROOT is None:
    raise FileNotFoundError('Workbook gabungan tidak ditemukan. Jalankan notebook dari paket proyek lengkap.')

OUTPUT_TABLE_DIR=PROJECT_ROOT/'outputs'/'tables'
OUTPUT_FIGURE_DIR=PROJECT_ROOT/'outputs'/'figures'
OUTPUT_MODEL_DIR=PROJECT_ROOT/'outputs'/'models'
for d in [OUTPUT_TABLE_DIR, OUTPUT_FIGURE_DIR, OUTPUT_MODEL_DIR]: d.mkdir(parents=True, exist_ok=True)

VALIDATED_STEM_FILE=OUTPUT_TABLE_DIR/'00_validated_AD_STEM_Jenjang.csv'
for readiness_name in ['00_readiness_checklist.csv','01_readiness_checklist.csv','02_readiness_checklist.csv']:
    rp=OUTPUT_TABLE_DIR/readiness_name
    if rp.exists():
        ready=pd.read_csv(rp)
        assert ready['status'].eq('PASS').all(), f'{readiness_name} belum seluruhnya PASS.'

print('CURRENT_DIR  :', CURRENT_DIR)
print('PROJECT_ROOT :', PROJECT_ROOT)
print('COMBINED_FILE:', COMBINED_FILE)

CURRENT_DIR  : /mnt/data/work_project04/pt_education_research_project
PROJECT_ROOT : /mnt/data/work_project04/pt_education_research_project
COMBINED_FILE: /mnt/data/work_project04/pt_education_research_project/data/processed/Dataset_Gabungan_Kemdiktisaintek_2025.xlsx


In [3]:
def standardize_columns(df):
    out=df.copy()
    out.columns=(pd.Index(out.columns).astype(str).str.strip().str.lower()
                 .str.replace(r'[^0-9a-zA-Z]+','_',regex=True)
                 .str.replace(r'_+','_',regex=True).str.strip('_'))
    return out


def safe_divide(a,b):
    a=pd.to_numeric(a,errors='coerce'); b=pd.to_numeric(b,errors='coerce')
    return a.div(b.where(b.ne(0)))


def recompute_indicators(df):
    out=df.copy()
    numeric=[
        'jumlah_lulusan','jumlah_mahasiswa_baru','mhs_baru_pt_akreditasi_tinggi',
        'mhs_baru_prodi_akreditasi_tinggi','mhs_baru_pt_prodi_tinggi','jumlah_pt',
        'jumlah_pt_akreditasi_tinggi','dosen_total_jabatan','dosen_profesor','dosen_senior',
        'dosen_total_sertifikasi','dosen_tersertifikasi','dosen_doktor_stem',
        'dosen_doktor_non_stem','dosen_doktor_total','dosen_doktor_sesuai_bidang'
    ]
    for col in numeric:
        if col in out: out[col]=pd.to_numeric(out[col],errors='coerce')
    out['flow_ratio']=safe_divide(out['jumlah_lulusan'],out['jumlah_mahasiswa_baru'])
    out['new_pt_high_share']=safe_divide(out['mhs_baru_pt_akreditasi_tinggi'],out['jumlah_mahasiswa_baru'])
    out['new_prodi_high_share']=safe_divide(out['mhs_baru_prodi_akreditasi_tinggi'],out['jumlah_mahasiswa_baru'])
    out['new_both_high_share']=safe_divide(out['mhs_baru_pt_prodi_tinggi'],out['jumlah_mahasiswa_baru'])
    out['pt_high_share']=safe_divide(out['jumlah_pt_akreditasi_tinggi'],out['jumlah_pt'])
    out['prof_share']=safe_divide(out['dosen_profesor'],out['dosen_total_jabatan'])
    out['senior_share']=safe_divide(out['dosen_senior'],out['dosen_total_jabatan'])
    out['cert_share']=safe_divide(out['dosen_tersertifikasi'],out['dosen_total_sertifikasi'])
    out['doctor_stem_share']=safe_divide(out['dosen_doktor_stem'],out['dosen_doktor_total'])
    out['log_pt']=np.log1p(out['jumlah_pt'])
    out['log_doctor_total']=np.log1p(out['dosen_doktor_total'])
    out['log_doctor_aligned']=np.log1p(out['dosen_doktor_sesuai_bidang'])
    return out


def zscore(df, cols):
    out=df.copy(); rows=[]
    for col in cols:
        mean=float(out[col].mean()); sd=float(out[col].std(ddof=0))
        assert np.isfinite(sd) and sd>0, f'SD tidak valid: {col}'
        out['z_'+col]=(out[col]-mean)/sd
        rows.append({'variable':col,'mean':mean,'std_population':sd,'minimum':float(out[col].min()),'maximum':float(out[col].max())})
    return out,pd.DataFrame(rows)


def gini(x):
    x=np.asarray(pd.Series(x).dropna(),dtype=float)
    x=x[x>=0]
    if len(x)==0 or x.sum()==0: return np.nan
    x=np.sort(x); n=len(x); idx=np.arange(1,n+1)
    return float((2*np.sum(idx*x)/(n*x.sum()))-(n+1)/n)


def theil(x):
    x=np.asarray(pd.Series(x).dropna(),dtype=float)
    x=x[x>=0]
    if len(x)==0 or x.mean()==0: return np.nan
    ratio=x/x.mean(); positive=ratio>0
    return float(np.mean(ratio[positive]*np.log(ratio[positive])))


def lorenz(x):
    x=np.sort(np.asarray(pd.Series(x).dropna(),dtype=float))
    x=x[x>=0]
    if len(x)==0 or x.sum()==0: return np.array([0,1]),np.array([0,1])
    return np.arange(len(x)+1)/len(x), np.r_[0,np.cumsum(x)/x.sum()]


def savefig(fig,name):
    path=OUTPUT_FIGURE_DIR/name
    fig.savefig(path,dpi=300,bbox_inches='tight')
    plt.close(fig)
    return path


def lr_test(reduced_name,reduced,full_name,full):
    stat=2*(full.llf-reduced.llf); df=int(round(full.df_model-reduced.df_model))
    p=float(stats.chi2.sf(stat,df)) if df>0 else np.nan
    return {'reduced_model':reduced_name,'full_model':full_name,'lr_statistic':float(stat),'df_difference':df,'p_value':p,'significant_0_05':bool(p<0.05) if np.isfinite(p) else False}


def grouped_binomial_fit(formula,df):
    X=patsy.dmatrix(formula,df,return_type='dataframe')
    y=np.column_stack([df['jumlah_lulusan_stem'],df['jumlah_lulusan_non_stem']])
    result=sm.GLM(y,X,family=sm.families.Binomial()).fit(
        cov_type='cluster',
        cov_kwds={'groups':df['provinsi_pt'],'use_correction':True}
    )
    return result,X


def fit_nb(formula,df,cov='cluster'):
    model=NegativeBinomial.from_formula(formula,df,offset=np.log(df['jumlah_mahasiswa_baru']),loglike_method='nb2')
    if cov=='cluster':
        return model.fit(disp=False,maxiter=500,cov_type='cluster',cov_kwds={'groups':df['provinsi_pt'],'use_correction':True})
    return model.fit(disp=False,maxiter=500,cov_type='HC1')


def coef_table(result,model,ratio_name='ratio'):
    ci=result.conf_int()
    out=pd.DataFrame({'model':model,'term':result.params.index,'coefficient':result.params.values,
                      'std_error':result.bse.values,'z_value':result.tvalues.values,'p_value':result.pvalues.values,
                      'ci_lower_coefficient':ci.iloc[:,0].values,'ci_upper_coefficient':ci.iloc[:,1].values})
    out[ratio_name]=np.exp(out['coefficient'])
    out[ratio_name+'_ci_lower']=np.exp(out['ci_lower_coefficient'])
    out[ratio_name+'_ci_upper']=np.exp(out['ci_upper_coefficient'])
    out['significant_0_05']=out['p_value']<0.05
    return out


def nb_model_row(name,r):
    return {'model':name,'nobs':int(r.nobs),'df_model':float(r.df_model),'log_likelihood':float(r.llf),
            'aic':float(r.aic),'bic':float(r.bic),'mcfadden_pseudo_r2':float(r.prsquared),
            'alpha':float(r.params.get('alpha',np.nan)),'converged':bool(r.mle_retvals.get('converged',False))}

## 1. Membaca data dan memvalidasi cakupan

Dataset utama adalah `AD_STEM_Jenjang`. Kunci uniknya adalah provinsi × kabupaten/kota × kategori STEM × jenjang.

In [4]:
if VALIDATED_STEM_FILE.exists():
    DATA_SOURCE='CSV tervalidasi Notebook 00'
    stem_df=pd.read_csv(VALIDATED_STEM_FILE)
else:
    DATA_SOURCE='Workbook gabungan — fallback'
    stem_df=pd.read_excel(COMBINED_FILE,sheet_name='AD_STEM_Jenjang',engine='openpyxl')

stem_df=recompute_indicators(standardize_columns(stem_df))
KEY=['provinsi_pt','kabupaten_pt','a_stem','nm_jenj_didik']
REQUIRED={*KEY,'jumlah_lulusan','jumlah_mahasiswa_baru','jumlah_pt','dosen_doktor_sesuai_bidang',
          'flag_match_mahasiswa_baru','flag_match_pt','flag_match_dosen_doktor','flag_match_dosen'}
missing=sorted(REQUIRED-set(stem_df.columns))
assert not missing, f'Kolom wajib hilang: {missing}'
assert not stem_df.duplicated(KEY).any(), 'Kunci AD_STEM_Jenjang tidak unik.'
assert set(stem_df['a_stem'].unique())=={'STEM','NON STEM'}
assert int(stem_df['jumlah_lulusan'].sum())==2547717

coverage=pd.DataFrame([
    {'indicator':'Jumlah baris','value':len(stem_df)},
    {'indicator':'Provinsi','value':stem_df['provinsi_pt'].nunique()},
    {'indicator':'Kabupaten/kota','value':stem_df['kabupaten_pt'].nunique()},
    {'indicator':'Total lulusan','value':stem_df['jumlah_lulusan'].sum()},
    {'indicator':'Baris mahasiswa baru tersedia','value':stem_df['jumlah_mahasiswa_baru'].notna().sum()},
    {'indicator':'Baris dosen doktor tersedia','value':stem_df['dosen_doktor_total'].notna().sum()},
])
coverage.to_csv(OUTPUT_TABLE_DIR/'04_data_coverage.csv',index=False)
display(coverage)
print('Data source:',DATA_SOURCE)

,indicator,value
0,Jumlah baris,2017
1,Provinsi,34
2,Kabupaten/kota,407
3,Total lulusan,2547717
4,Baris mahasiswa baru tersedia,1960
5,Baris dosen doktor tersedia,1780


Data source: Workbook gabungan — fallback


## 2. Profil nasional menurut STEM dan jenjang

Proporsi dihitung dari jumlah lulusan, bukan dari jumlah baris. Mahasiswa baru hanya digunakan sebagai pembanding komposisi aliran masuk.

In [5]:
national_category=(stem_df.groupby('a_stem',as_index=False)
                   .agg(jumlah_lulusan=('jumlah_lulusan','sum'),jumlah_mahasiswa_baru=('jumlah_mahasiswa_baru','sum')))
national_category['proporsi_lulusan']=national_category['jumlah_lulusan']/national_category['jumlah_lulusan'].sum()
national_category['proporsi_mahasiswa_baru']=national_category['jumlah_mahasiswa_baru']/national_category['jumlah_mahasiswa_baru'].sum()
national_category.to_csv(OUTPUT_TABLE_DIR/'04_national_category_summary.csv',index=False)

national_level=(stem_df.groupby(['nm_jenj_didik','a_stem'],as_index=False)
                .agg(jumlah_lulusan=('jumlah_lulusan','sum'),jumlah_mahasiswa_baru=('jumlah_mahasiswa_baru','sum')))
level_pivot=national_level.pivot(index='nm_jenj_didik',columns='a_stem',values=['jumlah_lulusan','jumlah_mahasiswa_baru']).fillna(0)
level_pivot.columns=['_'.join(map(str,c)).lower().replace(' ','_') for c in level_pivot.columns]
level_pivot=level_pivot.reset_index()
level_pivot['total_lulusan']=level_pivot['jumlah_lulusan_stem']+level_pivot['jumlah_lulusan_non_stem']
level_pivot['stem_graduate_share']=safe_divide(level_pivot['jumlah_lulusan_stem'],level_pivot['total_lulusan'])
level_pivot['total_mahasiswa_baru']=level_pivot['jumlah_mahasiswa_baru_stem']+level_pivot['jumlah_mahasiswa_baru_non_stem']
level_pivot['stem_new_share']=safe_divide(level_pivot['jumlah_mahasiswa_baru_stem'],level_pivot['total_mahasiswa_baru'])
level_pivot['composition_gap_pp']=(level_pivot['stem_graduate_share']-level_pivot['stem_new_share'])*100
level_pivot=level_pivot.sort_values('total_lulusan',ascending=False)
level_pivot.to_csv(OUTPUT_TABLE_DIR/'04_national_composition_by_level.csv',index=False)

display(national_category)
display(level_pivot)

fig,ax=plt.subplots(figsize=(11,5.5))
plot=level_pivot.sort_values('stem_graduate_share',ascending=False)
ax.bar(plot['nm_jenj_didik'],plot['stem_graduate_share']*100)
ax.set_ylabel('Proporsi lulusan STEM (%)')
ax.set_xlabel('Jenjang')
ax.set_title('Proporsi Lulusan STEM menurut Jenjang')
ax.tick_params(axis='x',rotation=45)
ax.grid(axis='y',alpha=.2)
savefig(fig,'04_01_stem_share_by_level.png')

,a_stem,jumlah_lulusan,jumlah_mahasiswa_baru,proporsi_lulusan,proporsi_mahasiswa_baru
0,NON STEM,1843757,"1,341,706.000000",0.723690,0.622418
1,STEM,703960,"813,928.000000",0.276310,0.377582


,nm_jenj_didik,jumlah_lulusan_non_stem,jumlah_lulusan_stem,jumlah_mahasiswa_baru_non_stem,jumlah_mahasiswa_baru_stem,total_lulusan,stem_graduate_share,total_mahasiswa_baru,stem_new_share,composition_gap_pp
5,S1,"858,814.000000","401,178.000000","1,123,074.000000","490,473.000000","1,259,992.000000",0.318397,"1,613,547.000000",0.303972,1.442533
4,Profesi,"812,319.000000","105,041.000000","27,511.000000","108,216.000000","917,360.000000",0.114504,"135,727.000000",0.797306,-68.280278
2,D3,"32,396.000000","108,482.000000","31,378.000000","108,198.000000","140,878.000000",0.770042,"139,576.000000",0.775191,-0.514841
6,S2,"106,218.000000","27,275.000000","108,147.000000","28,555.000000","133,493.000000",0.204318,"136,702.000000",0.208885,-0.456719
3,D4,"23,764.000000","51,538.000000","36,532.000000","65,603.000000","75,302.000000",0.684417,"102,135.000000",0.642317,4.210087
8,S3,"9,264.000000","2,838.000000","13,779.000000","4,500.000000","12,102.000000",0.234507,"18,279.000000",0.246184,-1.167745
9,Sp-1,0.000000,"4,124.000000",0.000000,"5,411.000000","4,124.000000",1.000000,"5,411.000000",1.000000,0.000000
0,D1,291.000000,"1,502.000000",344.000000,"1,121.000000","1,793.000000",0.837702,"1,465.000000",0.765188,7.251446
7,S2 Terapan,532.000000,718.000000,663.000000,471.000000,"1,250.000000",0.574400,"1,134.000000",0.415344,15.905608
1,D2,159.000000,799.000000,278.000000,852.000000,958.000000,0.834029,"1,130.000000",0.753982,8.004693


PosixPath('/mnt/data/work_project04/pt_education_research_project/outputs/figures/04_01_stem_share_by_level.png')

## 3. Profil STEM provinsi dan kabupaten/kota

`STEM location quotient` membandingkan proporsi STEM suatu wilayah dengan proporsi STEM nasional. Nilai di atas 1 menunjukkan spesialisasi relatif, bukan mutu atau efisiensi yang lebih tinggi.

`composition_gap_pp` adalah selisih proporsi STEM lulusan dan proporsi STEM mahasiswa baru pada periode yang sama. Ukuran ini bukan perubahan cohort.

In [6]:
def regional_profile(df,geo_cols):
    grouped=(df.groupby(geo_cols+['a_stem'],as_index=False)
             .agg(jumlah_lulusan=('jumlah_lulusan','sum'),jumlah_mahasiswa_baru=('jumlah_mahasiswa_baru','sum')))
    p=grouped.pivot(index=geo_cols,columns='a_stem',values=['jumlah_lulusan','jumlah_mahasiswa_baru'])
    p.columns=['_'.join(map(str,c)).lower().replace(' ','_') for c in p.columns]
    p=p.reset_index()
    for col in ['jumlah_lulusan_stem','jumlah_lulusan_non_stem','jumlah_mahasiswa_baru_stem','jumlah_mahasiswa_baru_non_stem']:
        if col not in p: p[col]=np.nan
    p['total_lulusan']=p[['jumlah_lulusan_stem','jumlah_lulusan_non_stem']].sum(axis=1,min_count=1)
    p['stem_graduate_share']=safe_divide(p['jumlah_lulusan_stem'],p['total_lulusan'])
    p['total_mahasiswa_baru']=p[['jumlah_mahasiswa_baru_stem','jumlah_mahasiswa_baru_non_stem']].sum(axis=1,min_count=1)
    p['stem_new_share']=safe_divide(p['jumlah_mahasiswa_baru_stem'],p['total_mahasiswa_baru'])
    p['composition_gap_pp']=(p['stem_graduate_share']-p['stem_new_share'])*100
    return p

national_stem_share=float(stem_df.loc[stem_df['a_stem'].eq('STEM'),'jumlah_lulusan'].sum()/stem_df['jumlah_lulusan'].sum())
province_profile=regional_profile(stem_df,['provinsi_pt'])
province_profile['stem_location_quotient']=province_profile['stem_graduate_share']/national_stem_share
province_profile=province_profile.sort_values('jumlah_lulusan_stem',ascending=False)
province_profile.to_csv(OUTPUT_TABLE_DIR/'04_province_stem_profile.csv',index=False)

district_profile=regional_profile(stem_df,['provinsi_pt','kabupaten_pt'])
district_profile['stem_location_quotient']=district_profile['stem_graduate_share']/national_stem_share
district_profile= district_profile.sort_values('jumlah_lulusan_stem',ascending=False)
district_profile.to_csv(OUTPUT_TABLE_DIR/'04_district_stem_profile.csv',index=False)

display(province_profile.head(15))

fig,ax=plt.subplots(figsize=(10,9))
plot=province_profile.sort_values('stem_graduate_share')
ax.barh(plot['provinsi_pt'],plot['stem_graduate_share']*100)
ax.axvline(national_stem_share*100,linestyle='--',linewidth=1,label='Nasional')
ax.set_xlabel('Proporsi lulusan STEM (%)')
ax.set_title('Komposisi Lulusan STEM menurut Provinsi')
ax.legend(); ax.grid(axis='x',alpha=.2)
savefig(fig,'04_02_province_stem_share.png')

fig,ax=plt.subplots(figsize=(10,9))
plot=province_profile.sort_values('stem_location_quotient')
ax.barh(plot['provinsi_pt'],plot['stem_location_quotient'])
ax.axvline(1,linestyle='--',linewidth=1)
ax.set_xlabel('STEM location quotient')
ax.set_title('Spesialisasi Relatif STEM menurut Provinsi')
ax.grid(axis='x',alpha=.2)
savefig(fig,'04_03_province_stem_location_quotient.png')

fig,ax=plt.subplots(figsize=(10,9))
plot=province_profile.dropna(subset=['composition_gap_pp']).sort_values('composition_gap_pp')
ax.barh(plot['provinsi_pt'],plot['composition_gap_pp'])
ax.axvline(0,linestyle='--',linewidth=1)
ax.set_xlabel('Proporsi STEM lulusan − mahasiswa baru (poin persentase)')
ax.set_title('Kesenjangan Komposisi Aliran STEM menurut Provinsi')
ax.grid(axis='x',alpha=.2)
savefig(fig,'04_04_province_stem_composition_gap.png')

,provinsi_pt,jumlah_lulusan_non_stem,jumlah_lulusan_stem,jumlah_mahasiswa_baru_non_stem,jumlah_mahasiswa_baru_stem,total_lulusan,stem_graduate_share,total_mahasiswa_baru,stem_new_share,composition_gap_pp,stem_location_quotient
11,Prov. Jawa Timur,"284,289.000000","100,535.000000","163,298.000000","107,219.000000","384,824.000000",0.261249,"270,517.000000",0.396348,-13.509918,0.945493
9,Prov. Jawa Barat,"197,029.000000","92,609.000000","138,952.000000","100,023.000000","289,638.000000",0.319741,"238,975.000000",0.418550,-9.880955,1.157180
10,Prov. Jawa Tengah,"202,510.000000","75,266.000000","105,091.000000","79,382.000000","277,776.000000",0.270959,"184,473.000000",0.430318,-15.935838,0.980635
6,Prov. D.K.I. Jakarta,"138,574.000000","66,752.000000","114,966.000000","70,094.000000","205,326.000000",0.325103,"185,060.000000",0.378764,-5.366112,1.176586
5,Prov. D.I. Yogyakarta,"105,055.000000","43,370.000000","53,311.000000","45,175.000000","148,425.000000",0.292201,"98,486.000000",0.458695,-16.649319,1.057513
33,Prov. Sumatera Utara,"99,150.000000","43,084.000000","57,760.000000","49,036.000000","142,234.000000",0.302909,"106,796.000000",0.459156,-15.624648,1.096266
27,Prov. Sulawesi Selatan,"92,697.000000","41,782.000000","49,843.000000","49,335.000000","134,479.000000",0.310695,"99,178.000000",0.497439,-18.674360,1.124444
31,Prov. Sumatera Barat,"62,806.000000","23,652.000000","34,082.000000","24,489.000000","86,458.000000",0.273566,"58,571.000000",0.418108,-14.454158,0.990070
3,Prov. Banten,"131,917.000000","21,354.000000","290,970.000000","48,947.000000","153,271.000000",0.139322,"339,917.000000",0.143997,-0.467506,0.504223
0,Prov. Aceh,"45,779.000000","17,225.000000","27,542.000000","21,092.000000","63,004.000000",0.273395,"48,634.000000",0.433688,-16.029303,0.989451


PosixPath('/mnt/data/work_project04/pt_education_research_project/outputs/figures/04_04_province_stem_composition_gap.png')

## 4. Ketimpangan distribusi keluaran STEM

Gini dan Theil dihitung terpisah untuk STEM dan non-STEM pada tingkat provinsi dan kabupaten/kota. Nilai mencerminkan konsentrasi volume, sehingga tetap dipengaruhi ukuran sistem pendidikan wilayah.

In [7]:
ineq=[]
for geography,profile in [('province',province_profile),('district',district_profile)]:
    for category,col in [('STEM','jumlah_lulusan_stem'),('NON STEM','jumlah_lulusan_non_stem')]:
        values=profile[col].fillna(0)
        ineq.append({'geography':geography,'category':category,'n_units':len(values),'total':float(values.sum()),
                     'gini':gini(values),'theil_t':theil(values),'top5_share':float(values.nlargest(5).sum()/values.sum())})
inequality=pd.DataFrame(ineq)
inequality.to_csv(OUTPUT_TABLE_DIR/'04_inequality_indices.csv',index=False)
display(inequality)

fig,ax=plt.subplots(figsize=(7,6))
for category,col in [('STEM','jumlah_lulusan_stem'),('NON STEM','jumlah_lulusan_non_stem')]:
    x,y=lorenz(province_profile[col].fillna(0)); ax.plot(x,y,label=category)
ax.plot([0,1],[0,1],linestyle='--',label='Kesetaraan sempurna')
ax.set_xlabel('Proporsi kumulatif provinsi')
ax.set_ylabel('Proporsi kumulatif lulusan')
ax.set_title('Kurva Lorenz Lulusan STEM dan Non-STEM')
ax.legend(); ax.grid(alpha=.2)
savefig(fig,'04_05_lorenz_province_stem_nonstem.png')

,geography,category,n_units,total,gini,theil_t,top5_share
0,province,STEM,34,"703,960.000000",0.581882,0.597137,0.537718
1,province,NON STEM,34,"1,843,757.000000",0.572872,0.570073,0.517595
2,district,STEM,414,"703,960.000000",0.825577,1.895994,0.219126
3,district,NON STEM,414,"1,843,757.000000",0.833799,1.657402,0.241189


PosixPath('/mnt/data/work_project04/pt_education_research_project/outputs/figures/04_05_lorenz_province_stem_nonstem.png')

## 5. Membentuk pasangan STEM–non-STEM

Satu baris pasangan mewakili provinsi × kabupaten/kota × jenjang yang memiliki kedua kategori. Prediktor tingkat wilayah diperiksa konsistensinya sebelum digabungkan.

In [8]:
PAIR_KEYS=['provinsi_pt','kabupaten_pt','nm_jenj_didik']
CATEGORY_VALUES=['jumlah_lulusan','jumlah_mahasiswa_baru','mhs_baru_pt_akreditasi_tinggi',
                 'mhs_baru_prodi_akreditasi_tinggi','mhs_baru_pt_prodi_tinggi','dosen_doktor_sesuai_bidang']
pair=stem_df.pivot_table(index=PAIR_KEYS,columns='a_stem',values=CATEGORY_VALUES,aggfunc='sum')
pair.columns=['_'.join(map(str,c)).lower().replace(' ','_') for c in pair.columns]
pair=pair.reset_index()

GEO_KEYS=['provinsi_pt','kabupaten_pt']
GEO_METRICS=['jumlah_pt','jumlah_pt_akreditasi_tinggi','dosen_total_jabatan','dosen_profesor','dosen_senior',
             'dosen_total_sertifikasi','dosen_tersertifikasi','dosen_doktor_stem','dosen_doktor_non_stem','dosen_doktor_total']
consistency=stem_df.groupby(GEO_KEYS)[GEO_METRICS].nunique(dropna=False)
assert not consistency.gt(1).any(axis=1).any(), 'Indikator wilayah tidak konsisten antarbaris.'
geo=stem_df[GEO_KEYS+GEO_METRICS].drop_duplicates(GEO_KEYS)
pair=pair.merge(geo,on=GEO_KEYS,how='left',validate='many_to_one')

pair['total_lulusan']=pair['jumlah_lulusan_stem']+pair['jumlah_lulusan_non_stem']
pair['stem_graduate_share']=safe_divide(pair['jumlah_lulusan_stem'],pair['total_lulusan'])
pair['total_mahasiswa_baru']=pair['jumlah_mahasiswa_baru_stem']+pair['jumlah_mahasiswa_baru_non_stem']
pair['stem_new_share']=safe_divide(pair['jumlah_mahasiswa_baru_stem'],pair['total_mahasiswa_baru'])
pair['stem_new_prodi_high_share']=safe_divide(pair['mhs_baru_prodi_akreditasi_tinggi_stem'],pair['jumlah_mahasiswa_baru_stem'])
pair['stem_new_pt_high_share']=safe_divide(pair['mhs_baru_pt_akreditasi_tinggi_stem'],pair['jumlah_mahasiswa_baru_stem'])
pair['pt_high_share']=safe_divide(pair['jumlah_pt_akreditasi_tinggi'],pair['jumlah_pt'])
pair['prof_share']=safe_divide(pair['dosen_profesor'],pair['dosen_total_jabatan'])
pair['cert_share']=safe_divide(pair['dosen_tersertifikasi'],pair['dosen_total_sertifikasi'])
pair['doctor_stem_share']=safe_divide(pair['dosen_doktor_stem'],pair['dosen_doktor_total'])
pair['log_doctor_total']=np.log1p(pair['dosen_doktor_total'])
pair['log_pt']=np.log1p(pair['jumlah_pt'])
pair['composition_gap_pp']=(pair['stem_graduate_share']-pair['stem_new_share'])*100

complete_pair=pair.dropna(subset=['jumlah_lulusan_stem','jumlah_lulusan_non_stem','jumlah_mahasiswa_baru_stem','jumlah_mahasiswa_baru_non_stem']).copy()
PAIR_CONT=['stem_new_share','stem_new_prodi_high_share','pt_high_share','doctor_stem_share','log_doctor_total','prof_share','cert_share']
model_pair=complete_pair.dropna(subset=PAIR_CONT).copy()
level_counts=model_pair['nm_jenj_didik'].value_counts()
eligible_levels=level_counts[level_counts>=20].index
model_pair=model_pair.loc[model_pair['nm_jenj_didik'].isin(eligible_levels)].copy()
model_pair,pair_scaling=zscore(model_pair,PAIR_CONT)

pair_flow=pd.DataFrame([
    {'stage':'Semua kombinasi wilayah–jenjang','n_rows':len(pair)},
    {'stage':'Pasangan STEM dan non-STEM lengkap','n_rows':len(complete_pair)},
    {'stage':'Complete-case prediktor','n_rows':len(complete_pair.dropna(subset=PAIR_CONT))},
    {'stage':'Jenjang dengan minimal 20 pasangan','n_rows':len(model_pair)},
])
pair_flow.to_csv(OUTPUT_TABLE_DIR/'04_paired_sample_flow.csv',index=False)
pair_scaling.to_csv(OUTPUT_TABLE_DIR/'04_binomial_scaling_parameters.csv',index=False)
model_pair.to_csv(OUTPUT_TABLE_DIR/'04_paired_modeling_dataset.csv',index=False)
display(pair_flow)
print('Eligible levels:',sorted(eligible_levels))

fig,ax=plt.subplots(figsize=(7,6))
weights=np.sqrt(model_pair['total_lulusan'])
ax.scatter(model_pair['stem_new_share']*100,model_pair['stem_graduate_share']*100,s=np.clip(weights,8,100),alpha=.35)
ax.plot([0,100],[0,100],linestyle='--')
ax.set_xlabel('Proporsi STEM mahasiswa baru (%)')
ax.set_ylabel('Proporsi STEM lulusan (%)')
ax.set_title('Komposisi STEM Mahasiswa Baru dan Lulusan')
ax.grid(alpha=.2)
savefig(fig,'04_06_entry_vs_graduate_stem_share.png')

,stage,n_rows
0,Semua kombinasi wilayah–jenjang,1352
1,Pasangan STEM dan non-STEM lengkap,665
2,Complete-case prediktor,614
3,Jenjang dengan minimal 20 pasangan,606


Eligible levels: ['D3', 'D4', 'Profesi', 'S1', 'S2', 'S3']


PosixPath('/mnt/data/work_project04/pt_education_research_project/outputs/figures/04_06_entry_vs_graduate_stem_share.png')

## 6. Model komposisi lulusan STEM

Hierarki model:

- B0: jenjang;
- B1: komposisi STEM mahasiswa baru;
- B2: akreditasi;
- B3: kapasitas doktor dan mutu tenaga akademik — model utama.

Karena terdapat overdispersion binomial, inferensi menggunakan standard error cluster-provinsi dan hasil dibaca secara hati-hati.

In [9]:
BIN_FORMULAS={
    'B0_level':'C(nm_jenj_didik, Treatment(reference="S1"))',
    'B1_entry_composition':'C(nm_jenj_didik, Treatment(reference="S1")) + z_stem_new_share',
    'B2_accreditation':'C(nm_jenj_didik, Treatment(reference="S1")) + z_stem_new_share + z_stem_new_prodi_high_share + z_pt_high_share',
    'B3_regional_capacity':'C(nm_jenj_didik, Treatment(reference="S1")) + z_stem_new_share + z_stem_new_prodi_high_share + z_pt_high_share + z_doctor_stem_share + z_log_doctor_total + z_prof_share + z_cert_share',
}
bin_results={}; bin_design={}; rows=[]
for name,formula in BIN_FORMULAS.items():
    result,X=grouped_binomial_fit(formula,model_pair)
    bin_results[name]=result; bin_design[name]=X
    rows.append({'model':name,'nobs':int(result.nobs),'df_model':float(result.df_model),'log_likelihood':float(result.llf),
                 'aic':float(result.aic),'deviance':float(result.deviance),
                 'pearson_dispersion':float(result.pearson_chi2/result.df_resid)})
bin_comparison=pd.DataFrame(rows).sort_values('aic').reset_index(drop=True)
bin_comparison['delta_aic']=bin_comparison['aic']-bin_comparison['aic'].min()
bin_comparison.to_csv(OUTPUT_TABLE_DIR/'04_binomial_model_comparison.csv',index=False)

bin_lr=[]
order=list(BIN_FORMULAS)
for a,b in zip(order[:-1],order[1:]): bin_lr.append(lr_test(a,bin_results[a],b,bin_results[b]))
bin_lr=pd.DataFrame(bin_lr)
bin_lr.to_csv(OUTPUT_TABLE_DIR/'04_binomial_likelihood_ratio_tests.csv',index=False)

display(bin_comparison); display(bin_lr)

BIN_PRIMARY='B3_regional_capacity'; BIN_RESULT=bin_results[BIN_PRIMARY]
bin_or=coef_table(BIN_RESULT,BIN_PRIMARY,'odds_ratio')
bin_or.to_csv(OUTPUT_TABLE_DIR/'04_binomial_primary_odds_ratios.csv',index=False)
display(bin_or.loc[~bin_or['term'].eq('Intercept'),['term','odds_ratio','odds_ratio_ci_lower','odds_ratio_ci_upper','p_value']])

,model,nobs,df_model,log_likelihood,aic,deviance,pearson_dispersion,delta_aic
0,B3_regional_capacity,606,12.000000,"-47,289.523401","94,605.046802","90,461.815810",171.344269,0.000000
1,B2_accreditation,606,8.000000,"-48,038.879408","96,095.758817","91,960.527825",176.657489,"1,490.712015"
2,B1_entry_composition,606,6.000000,"-49,938.659436","99,891.318872","95,760.087880",180.754403,"5,286.272070"
3,B0_level,606,5.000000,"-98,678.819301","197,369.638602","193,240.407610",335.633022,"102,764.591800"


,reduced_model,full_model,lr_statistic,df_difference,p_value,significant_0_05
0,B0_level,B1_entry_composition,"97,480.319730",1,0.000000,True
1,B1_entry_composition,B2_accreditation,"3,799.560055",2,0.000000,True
2,B2_accreditation,B3_regional_capacity,"1,498.712015",4,0.000000,True


,term,odds_ratio,odds_ratio_ci_lower,odds_ratio_ci_upper,p_value
1,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.D3]",1.346990,0.964583,1.881002,0.080409
2,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.D4]",1.364642,1.116301,1.668230,0.002418
3,"C(nm_jenj_didik, Treatment(reference=""S1""))[T....",0.038880,0.026733,0.056548,0.000000
4,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.S2]",0.799765,0.672720,0.950802,0.011356
5,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.S3]",0.840118,0.712392,0.990744,0.038407
6,z_stem_new_share,3.082031,2.408332,3.944187,0.000000
7,z_stem_new_prodi_high_share,1.147998,1.008669,1.306573,0.036553
8,z_pt_high_share,1.000710,0.946791,1.057700,0.979960
9,z_doctor_stem_share,1.104404,0.993595,1.227572,0.065644
10,z_log_doctor_total,0.963967,0.861114,1.079106,0.523819


In [10]:
model_pair['predicted_stem_share']=BIN_RESULT.predict(bin_design[BIN_PRIMARY])
model_pair['pearson_residual_binomial']=BIN_RESULT.resid_pearson
bin_metrics=pd.DataFrame([{
    'model':BIN_PRIMARY,'n':len(model_pair),
    'mae_unweighted_share':float(mean_absolute_error(model_pair['stem_graduate_share'],model_pair['predicted_stem_share'])),
    'rmse_unweighted_share':float(mean_squared_error(model_pair['stem_graduate_share'],model_pair['predicted_stem_share'])**.5),
    'weighted_brier_share':float(np.average((model_pair['stem_graduate_share']-model_pair['predicted_stem_share'])**2,weights=model_pair['total_lulusan'])),
    'pearson_dispersion':float(BIN_RESULT.pearson_chi2/BIN_RESULT.df_resid),
}])
bin_metrics.to_csv(OUTPUT_TABLE_DIR/'04_binomial_performance.csv',index=False)

rank=model_pair['predicted_stem_share'].rank(method='first')
model_pair['prediction_decile']=pd.qcut(rank,10,labels=False)+1
bin_cal=(model_pair.groupby('prediction_decile',as_index=False)
         .apply(lambda g: pd.Series({'n':len(g),'observed_share':np.average(g['stem_graduate_share'],weights=g['total_lulusan']),
                                     'predicted_share':np.average(g['predicted_stem_share'],weights=g['total_lulusan']),
                                     'graduates':g['total_lulusan'].sum()}),include_groups=False))
bin_cal.to_csv(OUTPUT_TABLE_DIR/'04_binomial_calibration_deciles.csv',index=False)

out_bin=model_pair.loc[model_pair['pearson_residual_binomial'].abs().nlargest(40).index,
    ['provinsi_pt','kabupaten_pt','nm_jenj_didik','jumlah_lulusan_stem','jumlah_lulusan_non_stem','stem_graduate_share','predicted_stem_share','pearson_residual_binomial']]
out_bin.to_csv(OUTPUT_TABLE_DIR/'04_binomial_residual_outliers.csv',index=False)

display(bin_metrics); display(bin_cal)

forest=bin_or.loc[~bin_or['term'].eq('Intercept')].sort_values('odds_ratio')
fig,ax=plt.subplots(figsize=(9,max(6,.38*len(forest))))
y=np.arange(len(forest)); err=np.vstack([forest['odds_ratio']-forest['odds_ratio_ci_lower'],forest['odds_ratio_ci_upper']-forest['odds_ratio']])
ax.errorbar(forest['odds_ratio'],y,xerr=err,fmt='o',capsize=3)
ax.axvline(1,linestyle='--',linewidth=1); ax.set_xscale('log')
ax.set_yticks(y); ax.set_yticklabels(forest['term']); ax.set_xlabel('Odds ratio (skala log)')
ax.set_title('Model Komposisi Lulusan STEM'); ax.grid(axis='x',alpha=.2)
savefig(fig,'04_07_binomial_odds_ratios.png')

fig,ax=plt.subplots(figsize=(7,6))
ax.plot(bin_cal['predicted_share']*100,bin_cal['observed_share']*100,marker='o')
lo=min(bin_cal['predicted_share'].min(),bin_cal['observed_share'].min())*100
hi=max(bin_cal['predicted_share'].max(),bin_cal['observed_share'].max())*100
ax.plot([lo,hi],[lo,hi],linestyle='--'); ax.set_xlabel('Prediksi proporsi STEM (%)'); ax.set_ylabel('Observasi proporsi STEM (%)')
ax.set_title('Kalibrasi Model Komposisi per Desil'); ax.grid(alpha=.2)
savefig(fig,'04_08_binomial_calibration.png')

,model,n,mae_unweighted_share,rmse_unweighted_share,weighted_brier_share,pearson_dispersion
0,B3_regional_capacity,606,0.078577,0.124370,0.005596,171.344269


,prediction_decile,n,observed_share,predicted_share,graduates
0,1,61.000000,0.082435,0.073811,"529,811.000000"
1,2,61.000000,0.118590,0.134943,"376,304.000000"
2,3,60.000000,0.125304,0.174690,"139,660.000000"
3,4,61.000000,0.250214,0.206710,"102,588.000000"
4,5,60.000000,0.288892,0.262041,"191,352.000000"
5,6,61.000000,0.334304,0.324403,"365,556.000000"
6,7,60.000000,0.382521,0.390983,"422,539.000000"
7,8,61.000000,0.525853,0.542468,"126,463.000000"
8,9,60.000000,0.740646,0.746487,"93,968.000000"
9,10,61.000000,0.888720,0.855917,"45,327.000000"


PosixPath('/mnt/data/work_project04/pt_education_research_project/outputs/figures/04_08_binomial_calibration.png')

## 7. Analisis sensitivitas model komposisi

- mengecualikan strata dengan total lulusan <100;
- menambahkan efek tetap provinsi;
- mengganti indikator akreditasi program studi STEM dengan akreditasi PT STEM.

In [11]:
# Sensitivitas disajikan sebagai koefisien utama terpilih.
def fit_bin_custom(formula,df): return grouped_binomial_fit(formula,df)[0]
small=model_pair.loc[model_pair['total_lulusan']>=100].copy()
formula_fe=BIN_FORMULAS[BIN_PRIMARY]+' + C(provinsi_pt)'
formula_pt=BIN_FORMULAS[BIN_PRIMARY].replace('z_stem_new_prodi_high_share','z_stem_new_pt_high_share')
# z STEM PT high perlu dibentuk pada sampel yang sama.
model_pair['z_stem_new_pt_high_share']=(model_pair['stem_new_pt_high_share']-model_pair['stem_new_pt_high_share'].mean())/model_pair['stem_new_pt_high_share'].std(ddof=0)
small['z_stem_new_pt_high_share']=(small['stem_new_pt_high_share']-model_pair['stem_new_pt_high_share'].mean())/model_pair['stem_new_pt_high_share'].std(ddof=0)

bin_sens={
    'B3_primary':BIN_RESULT,
    'S1_total_graduates_ge_100':fit_bin_custom(BIN_FORMULAS[BIN_PRIMARY],small),
    'S2_province_fixed_effects':fit_bin_custom(formula_fe,model_pair),
    'S3_stem_pt_accreditation':fit_bin_custom(formula_pt,model_pair),
}
selected=['z_stem_new_share','z_stem_new_prodi_high_share','z_stem_new_pt_high_share','z_doctor_stem_share','z_prof_share','z_cert_share']
parts=[]
for name,res in bin_sens.items():
    t=coef_table(res,name,'odds_ratio'); parts.append(t.loc[t['term'].isin(selected)])
bin_sensitivity=pd.concat(parts,ignore_index=True)
bin_sensitivity.to_csv(OUTPUT_TABLE_DIR/'04_binomial_sensitivity_selected_or.csv',index=False)
display(bin_sensitivity[['model','term','odds_ratio','odds_ratio_ci_lower','odds_ratio_ci_upper','p_value']])

,model,term,odds_ratio,odds_ratio_ci_lower,odds_ratio_ci_upper,p_value
0,B3_primary,z_stem_new_share,3.082031,2.408332,3.944187,0.000000
1,B3_primary,z_stem_new_prodi_high_share,1.147998,1.008669,1.306573,0.036553
2,B3_primary,z_doctor_stem_share,1.104404,0.993595,1.227572,0.065644
3,B3_primary,z_prof_share,0.946784,0.859393,1.043062,0.268423
4,B3_primary,z_cert_share,0.977798,0.910625,1.049925,0.536374
5,S1_total_graduates_ge_100,z_stem_new_share,3.082952,2.406023,3.950334,0.000000
6,S1_total_graduates_ge_100,z_stem_new_prodi_high_share,1.148138,1.008303,1.307365,0.037092
7,S1_total_graduates_ge_100,z_doctor_stem_share,1.104469,0.992943,1.228522,0.067314
8,S1_total_graduates_ge_100,z_prof_share,0.946827,0.859073,1.043546,0.270884
9,S1_total_graduates_ge_100,z_cert_share,0.977756,0.910547,1.049924,0.535831


## 8. Model Negative Binomial rasio aliran

Sampel memerlukan mahasiswa baru positif serta indikator akreditasi dan dosen lengkap. Jenjang dengan kurang dari 20 baris dikeluarkan untuk mencegah estimasi kategori sangat jarang.

Model:

- F1: kategori STEM dan jenjang;
- F2: tambahan akreditasi;
- F3: tambahan kapasitas doktor dan tenaga akademik;
- F4: dua interaksi teoritis — model utama.

Model offset-only tidak dipaksakan karena tidak stabil pada overdispersion ekstrem data ini.

In [12]:
FLOW_CONT=['new_pt_high_share','new_prodi_high_share','log_pt','pt_high_share','log_doctor_aligned','prof_share','cert_share']
flow_df=stem_df.dropna(subset=['jumlah_lulusan','jumlah_mahasiswa_baru','a_stem','nm_jenj_didik','provinsi_pt']+FLOW_CONT).copy()
flow_df=flow_df.loc[flow_df['jumlah_mahasiswa_baru']>0].copy()
level_count=flow_df['nm_jenj_didik'].value_counts(); flow_levels=level_count[level_count>=20].index
flow_df=flow_df.loc[flow_df['nm_jenj_didik'].isin(flow_levels)].copy()
flow_df,flow_scaling=zscore(flow_df,FLOW_CONT)
flow_df.to_csv(OUTPUT_TABLE_DIR/'04_flow_modeling_dataset.csv',index=False)
flow_scaling.to_csv(OUTPUT_TABLE_DIR/'04_flow_scaling_parameters.csv',index=False)
flow_sample=pd.DataFrame([
    {'stage':'Semua baris STEM dataset','n_rows':len(stem_df)},
    {'stage':'Complete-case dan mahasiswa baru positif','n_rows':len(stem_df.dropna(subset=['jumlah_lulusan','jumlah_mahasiswa_baru','a_stem','nm_jenj_didik','provinsi_pt']+FLOW_CONT).query('jumlah_mahasiswa_baru > 0'))},
    {'stage':'Jenjang minimal 20 observasi','n_rows':len(flow_df)},
])
flow_sample.to_csv(OUTPUT_TABLE_DIR/'04_flow_sample_flow.csv',index=False)
display(flow_sample)

BASE='C(a_stem, Treatment(reference="NON STEM")) + C(nm_jenj_didik, Treatment(reference="S1"))'
NB_FORMULAS={
    'F1_stem_level':'jumlah_lulusan ~ '+BASE,
    'F2_accreditation':'jumlah_lulusan ~ '+BASE+' + z_new_pt_high_share + z_new_prodi_high_share + z_log_pt + z_pt_high_share',
    'F3_regional_capacity':'jumlah_lulusan ~ '+BASE+' + z_new_pt_high_share + z_new_prodi_high_share + z_log_pt + z_pt_high_share + z_log_doctor_aligned + z_prof_share + z_cert_share',
    'F4_stem_interactions':'jumlah_lulusan ~ '+BASE+' + z_new_pt_high_share + z_new_prodi_high_share + z_log_pt + z_pt_high_share + z_log_doctor_aligned + z_prof_share + z_cert_share + C(a_stem, Treatment(reference="NON STEM")):z_new_prodi_high_share + C(a_stem, Treatment(reference="NON STEM")):z_log_doctor_aligned',
}
# Poisson diagnostic pada spesifikasi F3.
pois=smf.glm(NB_FORMULAS['F3_regional_capacity'],data=flow_df,family=sm.families.Poisson(),offset=np.log(flow_df['jumlah_mahasiswa_baru'])).fit()
poisson_diag=pd.DataFrame([{'pearson_chi2':float(pois.pearson_chi2),'df_resid':float(pois.df_resid),
                            'pearson_dispersion':float(pois.pearson_chi2/pois.df_resid),'deviance_per_df':float(pois.deviance/pois.df_resid)}])
poisson_diag.to_csv(OUTPUT_TABLE_DIR/'04_flow_poisson_diagnostics.csv',index=False)

nb_results={}; rows=[]
for name,formula in NB_FORMULAS.items():
    r=fit_nb(formula,flow_df); nb_results[name]=r; rows.append(nb_model_row(name,r))
nb_comparison=pd.DataFrame(rows).sort_values('aic').reset_index(drop=True)
nb_comparison['delta_aic']=nb_comparison['aic']-nb_comparison['aic'].min()
nb_comparison.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_model_comparison.csv',index=False)
nb_lr=[]; order=list(NB_FORMULAS)
for a,b in zip(order[:-1],order[1:]): nb_lr.append(lr_test(a,nb_results[a],b,nb_results[b]))
nb_lr=pd.DataFrame(nb_lr); nb_lr.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_likelihood_ratio_tests.csv',index=False)

display(poisson_diag); display(nb_comparison); display(nb_lr)
assert nb_comparison['converged'].all()

NB_PRIMARY='F4_stem_interactions'; NB_RESULT=nb_results[NB_PRIMARY]
nb_irr=coef_table(NB_RESULT,NB_PRIMARY,'irr')
nb_irr.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_primary_irr.csv',index=False)
display(nb_irr.loc[~nb_irr['term'].isin(['Intercept','alpha']),['term','irr','irr_ci_lower','irr_ci_upper','p_value']])

,stage,n_rows
0,Semua baris STEM dataset,2017
1,Complete-case dan mahasiswa baru positif,1722
2,Jenjang minimal 20 observasi,1698


,pearson_chi2,df_resid,pearson_dispersion,deviance_per_df
0,"2,858,845.784037","1,681.000000","1,700.681609",962.682122


,model,nobs,df_model,log_likelihood,aic,bic,mcfadden_pseudo_r2,alpha,converged,delta_aic
0,F4_stem_interactions,1698,18.000000,"-11,704.898231","23,449.796461","23,558.540589",0.101602,0.779912,True,0.000000
1,F3_regional_capacity,1698,16.000000,"-11,715.824774","23,467.649549","23,565.519263",0.100763,0.788810,True,17.853087
2,F2_accreditation,1698,13.000000,"-11,722.634081","23,475.268162","23,556.826257",0.100240,0.794545,True,25.471700
3,F1_stem_level,1698,9.000000,"-11,744.522411","23,511.044823","23,570.854093",0.098560,0.810507,True,61.248362


,reduced_model,full_model,lr_statistic,df_difference,p_value,significant_0_05
0,F1_stem_level,F2_accreditation,43.776661,4,0.000000,True
1,F2_accreditation,F3_regional_capacity,13.618613,3,0.003473,True
2,F3_regional_capacity,F4_stem_interactions,21.853087,2,0.000018,True


,term,irr,irr_ci_lower,irr_ci_upper,p_value
1,"C(a_stem, Treatment(reference=""NON STEM""))[T.S...",0.697742,0.627067,0.776382,0.000000
2,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.D2]",2.047633,1.271872,3.296558,0.003180
3,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.D3]",1.559662,1.432980,1.697545,0.000000
4,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.D4]",1.129123,0.968277,1.316687,0.121425
5,"C(nm_jenj_didik, Treatment(reference=""S1""))[T....",22.024410,14.275871,33.978637,0.000000
6,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.S2]",1.135171,0.926450,1.390915,0.221323
7,"C(nm_jenj_didik, Treatment(reference=""S1""))[T....",1.723478,1.217031,2.440673,0.002166
8,"C(nm_jenj_didik, Treatment(reference=""S1""))[T.S3]",0.710300,0.552628,0.912958,0.007561
9,"C(nm_jenj_didik, Treatment(reference=""S1""))[T....",0.931761,0.700503,1.239365,0.627258
10,z_new_pt_high_share,1.210995,0.946877,1.548784,0.127233


In [13]:
# Efek terpilih yang berbeda antara STEM dan non-STEM.
def combined_effect(result,base_term,interaction_term,label):
    names=list(result.params.index); cov=result.cov_params()
    b0=float(result.params[base_term]); v0=float(cov.loc[base_term,base_term])
    if interaction_term is None:
        coef=b0; var=v0
    else:
        bi=float(result.params[interaction_term]); coef=b0+bi
        var=v0+float(cov.loc[interaction_term,interaction_term])+2*float(cov.loc[base_term,interaction_term])
    se=math.sqrt(max(var,0)); lo=coef-1.96*se; hi=coef+1.96*se
    return {'effect':label,'coefficient':coef,'std_error':se,'irr':math.exp(coef),'irr_ci_lower':math.exp(lo),'irr_ci_upper':math.exp(hi),'p_value':float(2*stats.norm.sf(abs(coef/se))) if se>0 else np.nan}

stem_prefix='C(a_stem, Treatment(reference="NON STEM"))[T.STEM]:'
interaction_effects=pd.DataFrame([
    combined_effect(NB_RESULT,'z_new_prodi_high_share',None,'Akreditasi prodi tinggi — NON STEM'),
    combined_effect(NB_RESULT,'z_new_prodi_high_share',stem_prefix+'z_new_prodi_high_share','Akreditasi prodi tinggi — STEM'),
    combined_effect(NB_RESULT,'z_log_doctor_aligned',None,'Dosen doktor sesuai bidang — NON STEM'),
    combined_effect(NB_RESULT,'z_log_doctor_aligned',stem_prefix+'z_log_doctor_aligned','Dosen doktor sesuai bidang — STEM'),
])
interaction_effects.to_csv(OUTPUT_TABLE_DIR/'04_nb_interaction_combined_effects.csv',index=False)
display(interaction_effects)

fig,ax=plt.subplots(figsize=(9,5.5))
plot=nb_comparison.sort_values('aic',ascending=False)
ax.barh(plot['model'],plot['aic']); ax.set_xlabel('AIC (lebih rendah lebih baik)'); ax.set_title('Perbandingan Model Aliran Negative Binomial')
ax.grid(axis='x',alpha=.2)
savefig(fig,'04_09_nb_flow_aic_comparison.png')

forest=nb_irr.loc[~nb_irr['term'].isin(['Intercept','alpha'])].sort_values('irr')
fig,ax=plt.subplots(figsize=(10,max(7,.35*len(forest))))
y=np.arange(len(forest)); err=np.vstack([forest['irr']-forest['irr_ci_lower'],forest['irr_ci_upper']-forest['irr']])
ax.errorbar(forest['irr'],y,xerr=err,fmt='o',capsize=3); ax.axvline(1,linestyle='--',linewidth=1); ax.set_xscale('log')
ax.set_yticks(y); ax.set_yticklabels(forest['term']); ax.set_xlabel('IRR (skala log)'); ax.set_title('Model Aliran STEM/Non-STEM')
ax.grid(axis='x',alpha=.2)
savefig(fig,'04_10_nb_flow_irr.png')

fig,ax=plt.subplots(figsize=(8,5))
y=np.arange(len(interaction_effects)); err=np.vstack([interaction_effects['irr']-interaction_effects['irr_ci_lower'],interaction_effects['irr_ci_upper']-interaction_effects['irr']])
ax.errorbar(interaction_effects['irr'],y,xerr=err,fmt='o',capsize=3); ax.axvline(1,linestyle='--',linewidth=1); ax.set_xscale('log')
ax.set_yticks(y); ax.set_yticklabels(interaction_effects['effect']); ax.set_xlabel('IRR per 1 SD'); ax.set_title('Efek Gabungan Interaksi STEM')
ax.grid(axis='x',alpha=.2)
savefig(fig,'04_11_nb_interaction_effects.png')

,effect,coefficient,std_error,irr,irr_ci_lower,irr_ci_upper,p_value
0,Akreditasi prodi tinggi — NON STEM,-0.207598,0.080901,0.812533,0.693389,0.952151,0.010286
1,Akreditasi prodi tinggi — STEM,0.047150,0.041212,1.048279,0.966933,1.136469,0.252593
2,Dosen doktor sesuai bidang — NON STEM,0.195788,0.114585,1.216269,0.971613,1.522532,0.087511
3,Dosen doktor sesuai bidang — STEM,0.005001,0.050945,1.005014,0.909509,1.110547,0.921795


PosixPath('/mnt/data/work_project04/pt_education_research_project/outputs/figures/04_11_nb_interaction_effects.png')

## 9. Diagnostik model aliran dan residual regional

Kinerja bersifat in-sample. Generalisasi antarprovinsi harus dinilai pada Notebook 03/validasi GroupCV tersendiri.

In [14]:
flow_df['predicted_flow']=NB_RESULT.predict(flow_df,offset=np.log(flow_df['jumlah_mahasiswa_baru']))
alpha=float(NB_RESULT.params['alpha'])
flow_df['raw_residual']=flow_df['jumlah_lulusan']-flow_df['predicted_flow']
flow_df['pearson_residual']=flow_df['raw_residual']/np.sqrt(flow_df['predicted_flow']+alpha*flow_df['predicted_flow']**2)
flow_df['observed_predicted_ratio']=safe_divide(flow_df['jumlah_lulusan'],flow_df['predicted_flow'])

metrics=pd.DataFrame([{
    'model':NB_PRIMARY,'n':len(flow_df),'mae':float(mean_absolute_error(flow_df['jumlah_lulusan'],flow_df['predicted_flow'])),
    'rmse':float(mean_squared_error(flow_df['jumlah_lulusan'],flow_df['predicted_flow'])**.5),
    'rmsle':float(mean_squared_log_error(flow_df['jumlah_lulusan'],np.clip(flow_df['predicted_flow'],0,None))**.5),
    'mean_poisson_deviance':float(mean_poisson_deviance(flow_df['jumlah_lulusan'],np.clip(flow_df['predicted_flow'],1e-12,None))),
    'observed_total':float(flow_df['jumlah_lulusan'].sum()),'predicted_total':float(flow_df['predicted_flow'].sum()),
    'predicted_to_observed_ratio':float(flow_df['predicted_flow'].sum()/flow_df['jumlah_lulusan'].sum()),
    'alpha':alpha,
}])
metrics.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_in_sample_performance.csv',index=False)

out=flow_df.loc[flow_df['pearson_residual'].abs().nlargest(40).index,
    ['provinsi_pt','kabupaten_pt','a_stem','nm_jenj_didik','jumlah_lulusan','jumlah_mahasiswa_baru','predicted_flow','pearson_residual','observed_predicted_ratio']]
out.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_residual_outliers.csv',index=False)

province_res=(flow_df.groupby(['provinsi_pt','a_stem'],as_index=False)
              .agg(observed=('jumlah_lulusan','sum'),predicted=('predicted_flow','sum'),new_students=('jumlah_mahasiswa_baru','sum')))
province_res['residual']=province_res['observed']-province_res['predicted']
province_res['observed_to_predicted_ratio']=safe_divide(province_res['observed'],province_res['predicted'])
province_res.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_province_residuals.csv',index=False)
display(metrics); display(out.head(10))

fig,ax=plt.subplots(figsize=(7.5,6.5))
ax.scatter(flow_df['predicted_flow'],flow_df['jumlah_lulusan'],alpha=.35,s=18)
lo=min(flow_df['predicted_flow'].min(),flow_df['jumlah_lulusan'].min()); hi=max(flow_df['predicted_flow'].max(),flow_df['jumlah_lulusan'].max())
ax.plot([lo,hi],[lo,hi],linestyle='--'); ax.set_xscale('log'); ax.set_yscale('log')
ax.set_xlabel('Prediksi jumlah lulusan'); ax.set_ylabel('Observasi jumlah lulusan'); ax.set_title('Observed vs Predicted — Model Aliran NB')
ax.grid(alpha=.2)
savefig(fig,'04_12_nb_observed_vs_predicted.png')

provplot=province_res.copy(); provplot['label']=provplot['provinsi_pt']+' — '+provplot['a_stem']; provplot=provplot.sort_values('residual')
fig,ax=plt.subplots(figsize=(10,14))
ax.barh(provplot['label'],provplot['residual']); ax.axvline(0,linestyle='--',linewidth=1)
ax.set_xlabel('Observed minus predicted graduates'); ax.set_title('Residual Model Aliran menurut Provinsi dan Kategori')
ax.grid(axis='x',alpha=.2)
savefig(fig,'04_13_nb_province_category_residuals.png')

,model,n,mae,rmse,rmsle,mean_poisson_deviance,observed_total,predicted_total,predicted_to_observed_ratio,alpha
0,F4_stem_interactions,1698,"1,496.062299","7,146.588227",1.125873,"1,931.798269","2,414,956.000000","4,076,990.356063",1.688226,0.779912


,provinsi_pt,kabupaten_pt,a_stem,nm_jenj_didik,jumlah_lulusan,jumlah_mahasiswa_baru,predicted_flow,pearson_residual,observed_predicted_ratio
840,Prov. Jawa Timur,Kab. Bangkalan,NON STEM,Profesi,2139,1.000000,43.093095,54.271878,49.636722
1960,Prov. Sumatera Utara,Kab. Tapanuli Utara,NON STEM,Profesi,754,1.000000,29.777703,26.965196,25.320959
1526,Prov. Riau,Kab. Kampar,NON STEM,S2,227,8.000000,11.579357,19.988293,19.603853
1216,Prov. Kalimantan Utara,Kota Tarakan,NON STEM,Profesi,2113,5.000000,149.068557,14.854492,14.174686
1340,Prov. Maluku Utara,Kota Ternate,NON STEM,Profesi,4727,13.000000,410.851302,11.877155,11.505379
1992,Prov. Sumatera Utara,Kota Padang Sidempuan,NON STEM,Profesi,5506,23.000000,556.439652,10.060651,9.895053
1369,Prov. Nusa Tenggara Barat,Kab. Lombok Timur,STEM,D3,33,3.000000,2.909882,9.755485,11.340668
37,Prov. Aceh,Kab. Bireuen,NON STEM,Profesi,4971,22.000000,628.272816,7.818958,7.912168
1618,Prov. Sulawesi Selatan,Kab. Sidenreng Rappang,STEM,D4,100,18.000000,14.477890,6.410971,6.907084
597,Prov. Jawa Barat,Kota Cirebon,STEM,D4,163,42.000000,24.002856,6.388808,6.790859


PosixPath('/mnt/data/work_project04/pt_education_research_project/outputs/figures/04_13_nb_province_category_residuals.png')

## 10. Sensitivitas model aliran

- mengecualikan strata mahasiswa baru <100;
- menambahkan efek tetap provinsi dengan covariance HC1;
- membatasi jenjang pada enam kategori yang juga memenuhi sampel model komposisi.

In [15]:
flow_sens={
    'F4_primary':NB_RESULT,
    'S1_new_students_ge_100':fit_nb(NB_FORMULAS[NB_PRIMARY],flow_df.loc[flow_df['jumlah_mahasiswa_baru']>=100].copy()),
    'S2_province_fixed_effects':fit_nb(NB_FORMULAS[NB_PRIMARY]+' + C(provinsi_pt)',flow_df,cov='HC1'),
    'S3_common_levels_only':fit_nb(NB_FORMULAS[NB_PRIMARY],flow_df.loc[flow_df['nm_jenj_didik'].isin(eligible_levels)].copy()),
}
selected_terms=['C(a_stem, Treatment(reference="NON STEM"))[T.STEM]','z_new_prodi_high_share','z_log_doctor_aligned','z_cert_share']
parts=[]; summaries=[]
for name,res in flow_sens.items():
    summaries.append(nb_model_row(name,res)); t=coef_table(res,name,'irr'); parts.append(t.loc[t['term'].isin(selected_terms)])
flow_sens_summary=pd.DataFrame(summaries); flow_sens_summary.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_sensitivity_summary.csv',index=False)
flow_sens_terms=pd.concat(parts,ignore_index=True); flow_sens_terms.to_csv(OUTPUT_TABLE_DIR/'04_nb_flow_sensitivity_selected_irr.csv',index=False)
display(flow_sens_summary); display(flow_sens_terms[['model','term','irr','irr_ci_lower','irr_ci_upper','p_value']])

,model,nobs,df_model,log_likelihood,aic,bic,mcfadden_pseudo_r2,alpha,converged
0,F4_primary,1698,18.000000,"-11,704.898231","23,449.796461","23,558.540589",0.101602,0.779912,True
1,S1_new_students_ge_100,1184,18.000000,"-8,819.457955","17,678.915909","17,780.448986",0.077837,0.469613,True
2,S2_province_fixed_effects,1698,51.000000,"-11,677.271503","23,460.543006","23,748.714944",0.103722,0.759745,True
3,S3_common_levels_only,1627,15.000000,"-11,363.284899","22,760.569798","22,852.276181",0.100943,0.790700,True


,model,term,irr,irr_ci_lower,irr_ci_upper,p_value
0,F4_primary,"C(a_stem, Treatment(reference=""NON STEM""))[T.S...",0.697742,0.627067,0.776382,0.000000
1,F4_primary,z_new_prodi_high_share,0.812533,0.693391,0.952148,0.010286
2,F4_primary,z_log_doctor_aligned,1.216269,0.971617,1.522525,0.087511
3,F4_primary,z_cert_share,1.073507,1.031959,1.116729,0.000428
4,S1_new_students_ge_100,"C(a_stem, Treatment(reference=""NON STEM""))[T.S...",0.671377,0.614864,0.733083,0.000000
5,S1_new_students_ge_100,z_new_prodi_high_share,0.986672,0.874260,1.113538,0.827887
6,S1_new_students_ge_100,z_log_doctor_aligned,1.169896,1.021880,1.339353,0.022992
7,S1_new_students_ge_100,z_cert_share,1.016216,0.982604,1.050977,0.348576
8,S2_province_fixed_effects,"C(a_stem, Treatment(reference=""NON STEM""))[T.S...",0.685629,0.611897,0.768245,0.000000
9,S2_province_fixed_effects,z_new_prodi_high_share,0.808713,0.694624,0.941541,0.006213


## 11. Penyimpanan model, ringkasan, dan checklist

In [16]:
BIN_RESULT.save(OUTPUT_MODEL_DIR/'04_grouped_binomial_stem_composition.pkl')
NB_RESULT.save(OUTPUT_MODEL_DIR/'04_nb2_stem_flow_interactions.pkl')
flow_sens['S2_province_fixed_effects'].save(OUTPUT_MODEL_DIR/'04_nb2_stem_flow_province_fe.pkl')

with (OUTPUT_TABLE_DIR/'04_binomial_model_summary.txt').open('w',encoding='utf-8') as f: f.write(BIN_RESULT.summary().as_text())
with (OUTPUT_TABLE_DIR/'04_nb_flow_model_summary.txt').open('w',encoding='utf-8') as f: f.write(NB_RESULT.summary().as_text())

national_stem_graduates=int(stem_df.loc[stem_df['a_stem'].eq('STEM'),'jumlah_lulusan'].sum())
entry_term=bin_or.loc[bin_or['term'].eq('z_stem_new_share')].iloc[0]
stem_term=nb_irr.loc[nb_irr['term'].eq('C(a_stem, Treatment(reference="NON STEM"))[T.STEM]')].iloc[0]
summary={
    'data_source':DATA_SOURCE,'rows':len(stem_df),'national_stem_graduates':national_stem_graduates,
    'national_stem_share':national_stem_share,'paired_complete_rows':len(complete_pair),'binomial_model_rows':len(model_pair),
    'binomial_pearson_dispersion':float(BIN_RESULT.pearson_chi2/BIN_RESULT.df_resid),
    'stem_new_share_odds_ratio_per_sd':float(entry_term['odds_ratio']),
    'stem_new_share_or_ci_lower':float(entry_term['odds_ratio_ci_lower']),
    'stem_new_share_or_ci_upper':float(entry_term['odds_ratio_ci_upper']),
    'flow_model_rows':len(flow_df),'flow_poisson_dispersion':float(poisson_diag.loc[0,'pearson_dispersion']),
    'flow_nb_alpha':float(NB_RESULT.params['alpha']),'stem_vs_nonstem_flow_irr':float(stem_term['irr']),
    'stem_vs_nonstem_flow_ci_lower':float(stem_term['irr_ci_lower']),'stem_vs_nonstem_flow_ci_upper':float(stem_term['irr_ci_upper']),
    'best_binomial_model':BIN_PRIMARY,'best_flow_model':NB_PRIMARY,
}
with (OUTPUT_TABLE_DIR/'04_summary.json').open('w',encoding='utf-8') as f: json.dump(summary,f,ensure_ascii=False,indent=2)

findings=f"""# Ringkasan Hasil Notebook 04

- Terdapat **{national_stem_graduates:,} lulusan STEM**, yaitu **{national_stem_share*100:.2f}%** dari seluruh lulusan.
- Tersedia **{len(complete_pair):,}** pasangan wilayah–jenjang dengan STEM dan non-STEM lengkap; **{len(model_pair):,}** pasangan masuk model komposisi utama.
- Model komposisi menunjukkan bahwa kenaikan satu standar deviasi proporsi STEM mahasiswa baru berhubungan dengan odds komposisi lulusan STEM sebesar **{entry_term['odds_ratio']:.3f}** kali (95% CI {entry_term['odds_ratio_ci_lower']:.3f}–{entry_term['odds_ratio_ci_upper']:.3f}).
- Pearson dispersion model binomial sebesar **{BIN_RESULT.pearson_chi2/BIN_RESULT.df_resid:.2f}**, sehingga heterogenitas regional yang tidak teramati masih besar dan inferensi perlu dibaca secara hati-hati.
- Model aliran menggunakan **{len(flow_df):,}** strata. Poisson dispersion sebesar **{poisson_diag.loc[0,'pearson_dispersion']:.2f}** dan alpha NB2 sebesar **{NB_RESULT.params['alpha']:.3f}**, mendukung Negative Binomial.
- Pada nilai nol prediktor terstandardisasi, kategori STEM memiliki IRR aliran **{stem_term['irr']:.3f}** terhadap non-STEM (95% CI {stem_term['irr_ci_lower']:.3f}–{stem_term['irr_ci_upper']:.3f}); interpretasi akhir harus memperhitungkan interaksi yang signifikan.
- Semua hasil bersifat ekologis dan rasio lulusan/mahasiswa baru bukan completion rate.
"""
with (OUTPUT_TABLE_DIR/'04_key_findings.md').open('w',encoding='utf-8') as f: f.write(findings)
print(findings)

# Ringkasan Hasil Notebook 04

- Terdapat **703,960 lulusan STEM**, yaitu **27.63%** dari seluruh lulusan.
- Tersedia **665** pasangan wilayah–jenjang dengan STEM dan non-STEM lengkap; **606** pasangan masuk model komposisi utama.
- Model komposisi menunjukkan bahwa kenaikan satu standar deviasi proporsi STEM mahasiswa baru berhubungan dengan odds komposisi lulusan STEM sebesar **3.082** kali (95% CI 2.408–3.944).
- Pearson dispersion model binomial sebesar **171.34**, sehingga heterogenitas regional yang tidak teramati masih besar dan inferensi perlu dibaca secara hati-hati.
- Model aliran menggunakan **1,698** strata. Poisson dispersion sebesar **1700.68** dan alpha NB2 sebesar **0.780**, mendukung Negative Binomial.
- Pada nilai nol prediktor terstandardisasi, kategori STEM memiliki IRR aliran **0.698** terhadap non-STEM (95% CI 0.627–0.776); interpretasi akhir harus memperhitungkan interaksi yang signifikan.
- Semua hasil bersifat ekologis dan rasio lulusan/mahasiswa baru bukan com

In [17]:
expected_tables=[
'04_data_coverage.csv','04_national_category_summary.csv','04_national_composition_by_level.csv','04_province_stem_profile.csv',
'04_district_stem_profile.csv','04_inequality_indices.csv','04_paired_sample_flow.csv','04_paired_modeling_dataset.csv',
'04_binomial_model_comparison.csv','04_binomial_likelihood_ratio_tests.csv','04_binomial_primary_odds_ratios.csv',
'04_binomial_performance.csv','04_binomial_calibration_deciles.csv','04_binomial_residual_outliers.csv',
'04_binomial_sensitivity_selected_or.csv','04_flow_sample_flow.csv','04_flow_modeling_dataset.csv','04_flow_poisson_diagnostics.csv',
'04_nb_flow_model_comparison.csv','04_nb_flow_likelihood_ratio_tests.csv','04_nb_flow_primary_irr.csv',
'04_nb_interaction_combined_effects.csv','04_nb_flow_in_sample_performance.csv','04_nb_flow_residual_outliers.csv',
'04_nb_flow_province_residuals.csv','04_nb_flow_sensitivity_summary.csv','04_nb_flow_sensitivity_selected_irr.csv',
'04_summary.json','04_key_findings.md']
expected_figures=[f'04_{i:02d}_{name}' for i,name in [
(1,'stem_share_by_level.png'),(2,'province_stem_share.png'),(3,'province_stem_location_quotient.png'),
(4,'province_stem_composition_gap.png'),(5,'lorenz_province_stem_nonstem.png'),(6,'entry_vs_graduate_stem_share.png'),
(7,'binomial_odds_ratios.png'),(8,'binomial_calibration.png'),(9,'nb_flow_aic_comparison.png'),
(10,'nb_flow_irr.png'),(11,'nb_interaction_effects.png'),(12,'nb_observed_vs_predicted.png'),
(13,'nb_province_category_residuals.png')]]
expected_models=['04_grouped_binomial_stem_composition.pkl','04_nb2_stem_flow_interactions.pkl','04_nb2_stem_flow_province_fe.pkl']

checks=pd.DataFrame([
{'check':'Kunci data unik','status':'PASS' if not stem_df.duplicated(KEY).any() else 'FAIL','evidence':f'{len(stem_df)} baris'},
{'check':'Cakupan provinsi','status':'PASS' if stem_df.provinsi_pt.nunique()>=30 else 'FAIL','evidence':f'{stem_df.provinsi_pt.nunique()} provinsi'},
{'check':'Pasangan STEM/non-STEM memadai','status':'PASS' if len(model_pair)>=500 else 'FAIL','evidence':f'{len(model_pair)} pasangan model'},
{'check':'Model binomial selesai','status':'PASS' if np.isfinite(BIN_RESULT.aic) else 'FAIL','evidence':f'AIC={BIN_RESULT.aic:.2f}'},
{'check':'Overdispersion binomial terdokumentasi','status':'PASS' if BIN_RESULT.pearson_chi2/BIN_RESULT.df_resid>1 else 'FAIL','evidence':f'{BIN_RESULT.pearson_chi2/BIN_RESULT.df_resid:.2f}'},
{'check':'Seluruh model NB konvergen','status':'PASS' if nb_comparison.converged.all() else 'FAIL','evidence':f'{nb_comparison.converged.sum()}/{len(nb_comparison)}'},
{'check':'Alpha NB positif','status':'PASS' if NB_RESULT.params['alpha']>0 else 'FAIL','evidence':f"{NB_RESULT.params['alpha']:.4f}"},
{'check':'Tabel lengkap','status':'PASS' if all((OUTPUT_TABLE_DIR/x).exists() for x in expected_tables) else 'FAIL','evidence':f"{sum((OUTPUT_TABLE_DIR/x).exists() for x in expected_tables)}/{len(expected_tables)}"},
{'check':'Figur lengkap','status':'PASS' if all((OUTPUT_FIGURE_DIR/x).exists() for x in expected_figures) else 'FAIL','evidence':f"{sum((OUTPUT_FIGURE_DIR/x).exists() for x in expected_figures)}/{len(expected_figures)}"},
{'check':'Model tersimpan','status':'PASS' if all((OUTPUT_MODEL_DIR/x).exists() for x in expected_models) else 'FAIL','evidence':f"{sum((OUTPUT_MODEL_DIR/x).exists() for x in expected_models)}/{len(expected_models)}"},
])
checks.to_csv(OUTPUT_TABLE_DIR/'04_readiness_checklist.csv',index=False)
display(checks)
assert checks['status'].eq('PASS').all(), 'Notebook 04 belum seluruhnya PASS.'

manifest=[]
for folder,kind in [(OUTPUT_TABLE_DIR,'table'),(OUTPUT_FIGURE_DIR,'figure'),(OUTPUT_MODEL_DIR,'model')]:
    for p in sorted(folder.glob('04_*')):
        if p.is_file(): manifest.append({'artifact_type':kind,'filename':p.name,'relative_path':str(p.relative_to(PROJECT_ROOT)),'size_bytes':p.stat().st_size})
pd.DataFrame(manifest).to_csv(OUTPUT_TABLE_DIR/'04_output_manifest.csv',index=False)

print('='*78)
print('NOTEBOOK 04 LOLOS.')
print('Analisis STEM regional selesai dan siap digunakan pada penyusunan artikel.')
print('='*78)

,check,status,evidence
0,Kunci data unik,PASS,2017 baris
1,Cakupan provinsi,PASS,34 provinsi
2,Pasangan STEM/non-STEM memadai,PASS,606 pasangan model
3,Model binomial selesai,PASS,AIC=94605.05
4,Overdispersion binomial terdokumentasi,PASS,171.34
5,Seluruh model NB konvergen,PASS,4/4
6,Alpha NB positif,PASS,0.7799
7,Tabel lengkap,PASS,29/29
8,Figur lengkap,PASS,13/13
9,Model tersimpan,PASS,3/3


NOTEBOOK 04 LOLOS.
Analisis STEM regional selesai dan siap digunakan pada penyusunan artikel.
